# 09. Benchmark against upstream

Is NeoProphet faster or more accurate than the Prophet it was forked from? This notebook measures it. One runner script is executed against upstream Prophet and against NeoProphet's default, tuned and MLX variants on the same datasets and holdouts, and the results are compared on forecast accuracy, speed, tuning cost and many-series throughput. Every number and every conclusion below is computed from the run, so the notebook follows the current code. It takes about ten minutes and needs network access, to build the upstream environment.

- What is measured, and the caveats
- Why subprocesses?
- Setup
- Run the benchmarks
- Environment and versions
- Accuracy on the holdout
- Speed
- Forecast overlays
- Summary of the default comparison
- Accuracy, speed and cost of all four variants
- Many-series throughput
- Summary of the fork's additions

Previous: [08. Many series on Apple Silicon](08_many_series_mlx.ipynb) | Back to the start: [00. Quick start](00_quick_start.ipynb)

In [ ]:
import json
import logging
import os
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from helpers import quiet_logs, repo_root

quiet_logs()

## What is measured

NeoProphet, the fork in this repo, is compared with upstream Facebook Prophet from PyPI on **speed** and **holdout forecast accuracy**, and then the notebook measures what the fork adds on top of upstream.

- The example datasets from `examples/`, with the per-dataset Prophet settings used in `benchmark_all_datasets.ipynb`.
- For each dataset the final part of the series is held out (the horizon depends on the series, and section 4 lists it). The model is fitted on the rest and predicts the held-out timestamps with uncertainty intervals.
- Accuracy: MAE, RMSE, sMAPE, MASE (scaled by the in-sample seasonal-naive error), 80% interval coverage and mean interval width.
- Speed: fit time and predict time (with uncertainty), median of several repeats after one untimed warm-up fit. A sequential `cross_validation` run on one dataset is timed as well.

**Four variants**

| Variant | What runs |
| --- | --- |
| Upstream | upstream Prophet from PyPI, the dataset's hand-set settings |
| NeoProphet default | the fork with the same settings and the same call as upstream |
| NeoProphet tuned | `prophet.tuning.tune` on the **training part only**, then the best model forecasts the holdout. Tuning time is reported separately. |
| NeoProphet MLX | `Prophet(stan_backend="MLX")` (experimental, linear and flat growth, MAP only) |

Sections 1 to 7 compare upstream with NeoProphet's default settings. Sections 8 to 11 add the tuned and MLX variants, the tuning cost and a many-series throughput scenario (CmdStan one by one against the MLX `fit_many`, which [08. Many series on Apple Silicon](08_many_series_mlx.ipynb) explains).

**Caveats**

- The fork is measured at whatever state `python/prophet/` is in when the notebook runs. The upstream side is pinned by `UPSTREAM_SPEC` in the setup cell, and the versions that actually ran are read back from each run's metadata in section 3.
- Timings come from a single machine and depend on background load, so treat small differences as noise. Only large, consistent gaps are meaningful.
- Seeds are set identically before every `predict`, but identical seeds only make the sampled intervals comparable when both versions draw random numbers in the same order. Point forecasts (`yhat`) do not depend on the seed.
- A handful of datasets with one holdout each is a small sample. This is not a statistical accuracy study. In particular, whether tuning helps on a given holdout is partly luck of that one window.
- The tuned variant searches a grid over `changepoint_prior_scale`, `seasonality_prior_scale` and `seasonality_mode`, so it discards the hand-set values of those three parameters. The hand-set settings are not guaranteed to be worse than the grid, and the grid's own default point is Prophet's defaults, not the hand-set values.
- MLX is an experimental optimiser (float32 on the GPU by default); its fits are close to, but not identical with, CmdStan's.

## Why subprocesses?

Both packages import as `prophet`, so they cannot be installed in one environment. A single runner script, `benchmarks/compare_upstream.py`, is executed in separate processes so every side is measured by identical code:

- **NeoProphet:** `uv run --directory <repo> python ...` uses the project environment (editable install of `python/`). It is run three times with `--variant default`, `tuned` and `mlx`.
- **Upstream:** `uv run --isolated --no-project --with <UPSTREAM_SPEC> python -I ...` builds a throwaway environment containing only the upstream PyPI release. `-I` keeps the current directory and `PYTHONPATH` off `sys.path`, so the fork cannot leak in. Only `--variant default` is requested here; the runner refuses `tuned` and `mlx` when the installed prophet lacks them.

Each run writes a JSON file to a temporary directory outside the repo, and this notebook loads the files. The runs are sequential, never concurrent, so they do not compete for CPU. The tuned variant uses a process pool of its own, so its tuning time reflects all CPU cores.

## 1. Setup

In [ ]:
logging.getLogger("matplotlib").setLevel(logging.WARNING)

ROOT = repo_root()
RUNNER = ROOT / "benchmarks" / "compare_upstream.py"
OUT_DIR = Path(tempfile.mkdtemp(prefix="neoprophet_vs_upstream_"))

REPEATS = 3
SEED = 42
# The pinned upstream release. The version that actually ran is read from the run's metadata.
UPSTREAM_SPEC = "prophet==1.5.0"
UPSTREAM_PYTHON = "3.13"
# Tuning is deterministic for a fixed seed, so one run per dataset gives the parameters;
# raise this only to tighten the tuning wall time.
TUNE_REPEATS = 1
THROUGHPUT_SIZES = "10,100"
THROUGHPUT_REPEATS = 3

FORK, UPSTREAM = "NeoProphet default", "Upstream"
TUNED, MLX = "NeoProphet tuned", "NeoProphet MLX"
ORDER = [UPSTREAM, FORK, TUNED, MLX]
COLORS = {
    FORK: "#1b6ca8",
    UPSTREAM: "#e07b00",
    TUNED: "#2a9d5c",
    MLX: "#8e44ad",
    "actual": "#333333",
}

pd.set_option("display.float_format", "{:,.4g}".format)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 10,
    }
)
print(f"Repo root: {ROOT}")
print(f"Results:   {OUT_DIR}")

## 2. Run the benchmarks

Four sequential runs. The first upstream run also downloads and caches the wheel, which is not included in any timing. The tuned run is the slow one because it cross-validates the whole default parameter grid for every dataset (section 9 reports the grid size). The throughput scenario fits seeded, perturbed copies of Peyton Manning (the sizes are set by `THROUGHPUT_SIZES`), one Prophet per series with CmdStan, and for MLX also with `fit_many`.

In [ ]:
common = [
    str(RUNNER),
    "--examples-dir", str(ROOT / "examples"),
    "--repeats", str(REPEATS),
    "--seed", str(SEED),
]
throughput = ["--throughput", THROUGHPUT_SIZES, "--throughput-repeats", str(THROUGHPUT_REPEATS)]
fork_run = ["uv", "run", "--directory", str(ROOT), "python", *common]
upstream_run = [
    "uv", "run", "--isolated", "--no-project", "--python", UPSTREAM_PYTHON,
    "--with", UPSTREAM_SPEC, "python", "-I", *common,
]

files = {label: OUT_DIR / f"{label.lower().replace(' ', '_')}.json" for label in ORDER}
commands = {
    FORK: [*fork_run, "--label", FORK, "--variant", "default", *throughput, "--output", str(files[FORK])],
    UPSTREAM: [*upstream_run, "--label", UPSTREAM, "--variant", "default", *throughput, "--output", str(files[UPSTREAM])],
    TUNED: [
        *fork_run, "--label", TUNED, "--variant", "tuned", "--tune-repeats", str(TUNE_REPEATS),
        "--skip-cv", "--output", str(files[TUNED]),
    ],
    MLX: [*fork_run, "--label", MLX, "--variant", "mlx", *throughput, "--skip-cv", "--output", str(files[MLX])],
}

# The kernel exports an inline matplotlib backend that a plain interpreter cannot load.
child_env = {**os.environ, "MPLBACKEND": "Agg"}

for label, cmd in commands.items():
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False, env=child_env)
    log_lines = [ln for ln in proc.stderr.splitlines() if "INFO" in ln and "repeat" not in ln]
    print(f"{label}: exit code {proc.returncode}")
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(f"{label} benchmark failed")
    print("  " + "\n  ".join(log_lines[-3:]))
    # cmdstanpy logs a fall-back to a second optimiser at ERROR level; count rather than hide it.
    fallbacks = sum("Falling back to Newton" in ln for ln in proc.stderr.splitlines())
    if fallbacks:
        print(f"  note: {fallbacks} fits fell back from L-BFGS to Newton (see stderr of the run)")

## 3. Environment and versions

In [ ]:
runs = {label: json.loads(files[label].read_text()) for label in ORDER}

env = pd.DataFrame({label: run["metadata"] for label, run in runs.items()}).drop(index="label")
env.loc["prophet_path"] = env.loc["prophet_path"].str.replace(str(Path.home()), "~", regex=False)
env

## 4. Accuracy on the holdout

Lower is better for MAE, RMSE, sMAPE (in %) and MASE. Coverage should sit near the nominal 0.80 and a narrower interval is better only while coverage holds. The delta is NeoProphet minus upstream.

In [ ]:
METRICS = ["mae", "rmse", "smape", "mase", "coverage", "interval_width"]


def metrics_frame(run: dict) -> pd.DataFrame:
    return pd.DataFrame({d["name"]: d["metrics"] for d in run["datasets"]}).T[METRICS]


fork_m, up_m = metrics_frame(runs[FORK]), metrics_frame(runs[UPSTREAM])
datasets = list(fork_m.index)

side_by_side = pd.concat(
    {FORK: fork_m, UPSTREAM: up_m, "delta": fork_m - up_m,
     "delta %": 100 * (fork_m - up_m) / up_m.abs().replace(0, np.nan)},
    axis=1,
).swaplevel(axis=1)
side_by_side = side_by_side.reindex(
    columns=pd.MultiIndex.from_product([METRICS, [FORK, UPSTREAM, "delta", "delta %"]])
)

side_by_side

### Context for the metrics

In [ ]:
spec = pd.DataFrame(
    {
        d["name"]: {
            "train rows": d["n_train"],
            "holdout steps": d["horizon"],
            "MASE season": d["season"],
        }
        for d in runs[FORK]["datasets"]
    }
).T
spec

## 5. Speed

Fit and predict times are medians over the repeats. The time axis is logarithmic because the datasets range from a few hundred to many thousands of rows, so their times differ by orders of magnitude.

In [ ]:
def timing_frame(run: dict) -> pd.DataFrame:
    return pd.DataFrame({d["name"]: d["timings_median"] for d in run["datasets"]}).T


fork_t, up_t = timing_frame(runs[FORK]), timing_frame(runs[UPSTREAM])

speed = pd.concat({FORK: fork_t, UPSTREAM: up_t}, axis=1)
speed[("speed-up (upstream / NeoProphet)", "total_s")] = up_t["total_s"] / fork_t["total_s"]
speed

In [ ]:
x = np.arange(len(datasets))
width = 0.38
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=False)

for ax, col, title in zip(axes, ["fit_s", "predict_s"], ["Fit time", "Predict time (with uncertainty)"]):
    ax.bar(x - width / 2, fork_t[col], width, label=FORK, color=COLORS[FORK])
    ax.bar(x + width / 2, up_t[col], width, label=UPSTREAM, color=COLORS[UPSTREAM])
    ax.set_yscale("log")
    ax.set_xticks(x, datasets, rotation=35, ha="right")
    ax.set_ylabel("seconds (median, log scale)")
    ax.set_title(title)
axes[0].legend(frameon=False)
fig.suptitle("Per-dataset timing", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
ratio = (up_t["total_s"] / fork_t["total_s"]).rename("speed-up")
geo_mean = float(np.exp(np.log(ratio).mean()))
sum_ratio = float(up_t["total_s"].sum() / fork_t["total_s"].sum())

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [3, 1.3]})
bar_colors = [COLORS[FORK] if r >= 1 else COLORS[UPSTREAM] for r in ratio]
ax.bar(datasets, ratio, color=bar_colors)
ax.axhline(1, color=COLORS["actual"], lw=1, ls="--")
ax.set_ylabel("upstream time / NeoProphet time")
ax.set_title("Total (fit + predict) speed-up per dataset\nabove 1 means NeoProphet is faster")
ax.tick_params(axis="x", rotation=35)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")

agg = pd.Series({"geometric mean\nof ratios": geo_mean, "total time\n(sum of datasets)": sum_ratio})
ax2.bar(agg.index, agg.values, color="#6a6a6a")
ax2.axhline(1, color=COLORS["actual"], lw=1, ls="--")
ax2.set_title("Aggregate speed-up")
for i, v in enumerate(agg.values):
    ax2.text(i, v, f"{v:.2f}x", ha="center", va="bottom")
fig.tight_layout()
plt.show()

### Cross-validation timing

In [ ]:
cv = pd.DataFrame(
    {label: {k: v for k, v in run["cross_validation"].items() if k != "durations_s"}
     | {"all_durations_s": [round(x, 2) for x in run["cross_validation"]["durations_s"]]}
     for label, run in runs.items() if run["cross_validation"]}
)
cv_speedup = runs[UPSTREAM]["cross_validation"]["median_s"] / runs[FORK]["cross_validation"]["median_s"]
display(cv)
print(f"cross_validation speed-up (upstream / NeoProphet): {cv_speedup:.2f}x")

## 6. Forecast overlays

Solid lines and filled bands are NeoProphet, dashed lines and dotted band edges are upstream. When the two packages produce the same forecast the curves coincide, so the bottom row of the figure also plots the absolute difference in `yhat`.

In [ ]:
SHOWN = ["Peyton Manning", "Air Passengers", "Pedestrians COVID", "Yosemite Temps"]


def forecast_frame(run: dict, name: str) -> pd.DataFrame:
    d = next(d for d in run["datasets"] if d["name"] == name)
    return pd.DataFrame(d["forecast"]).assign(ds=lambda f: pd.to_datetime(f["ds"]))


fig, axes = plt.subplots(2, len(SHOWN), figsize=(5 * len(SHOWN), 6.5),
                         gridspec_kw={"height_ratios": [3, 1]})
for col, name in enumerate(SHOWN):
    f, u = forecast_frame(runs[FORK], name), forecast_frame(runs[UPSTREAM], name)
    ax, ax_diff = axes[0, col], axes[1, col]
    ax.fill_between(f["ds"], f["yhat_lower"], f["yhat_upper"], color=COLORS[FORK], alpha=0.18, lw=0)
    ax.plot(u["ds"], u["yhat_lower"], color=COLORS[UPSTREAM], ls=":", lw=1)
    ax.plot(u["ds"], u["yhat_upper"], color=COLORS[UPSTREAM], ls=":", lw=1)
    ax.plot(f["ds"], f["y"], color=COLORS["actual"], lw=1.2, label="Actual")
    ax.plot(f["ds"], f["yhat"], color=COLORS[FORK], lw=1.8, label=FORK)
    ax.plot(u["ds"], u["yhat"], color=COLORS[UPSTREAM], lw=1.4, ls="--", label=UPSTREAM)
    ax.set_title(name)
    ax.set_ylabel("y")
    ax.tick_params(axis="x", labelbottom=False)

    ax_diff.plot(f["ds"], (f["yhat"] - u["yhat"]).abs(), color="#6a6a6a", lw=1)
    ax_diff.set_ylabel("|yhat diff|")
    ax_diff.set_xlabel("holdout date")
    ax_diff.tick_params(axis="x", rotation=30)
axes[0, 0].legend(frameon=False, loc="best")
fig.tight_layout()
plt.show()

## 7. Summary of the default comparison

In [ ]:
def pct(x: float) -> str:
    return f"{x:+.1f}%"


max_yhat_diff = {
    name: float((forecast_frame(runs[FORK], name)["yhat"] - forecast_frame(runs[UPSTREAM], name)["yhat"]).abs().max())
    for name in datasets
}
rel_metric_change = (100 * (fork_m - up_m) / up_m.abs().replace(0, np.nan)).abs()
better_mase = (fork_m["mase"] < up_m["mase"] - 1e-9).sum()
worse_mase = (fork_m["mase"] > up_m["mase"] + 1e-9).sum()
faster = int((ratio > 1).sum())

fv, uv = runs[FORK]["metadata"]["prophet_version"], runs[UPSTREAM]["metadata"]["prophet_version"]
lines = [
    f"Compared NeoProphet (prophet {fv}) with upstream prophet {uv} on {len(datasets)} datasets, "
    f"median of {REPEATS} repeats, on {runs[FORK]['metadata']['cpu']}.",
    "",
    "Accuracy",
    f"- Largest absolute yhat difference on any holdout point: {max(max_yhat_diff.values()):.3g} "
    f"({max(max_yhat_diff, key=max_yhat_diff.get)}).",
    f"- Largest relative change in any accuracy metric: {rel_metric_change.max().max():.3g}%.",
    f"- MASE better for NeoProphet on {better_mase} datasets, worse on {worse_mase}, "
    f"unchanged on {len(datasets) - better_mase - worse_mase}.",
    f"- Mean MASE: NeoProphet {fork_m['mase'].mean():.4f}, upstream {up_m['mase'].mean():.4f}.",
    f"- Mean interval coverage: NeoProphet {fork_m['coverage'].mean():.3f}, upstream {up_m['coverage'].mean():.3f} (nominal 0.80).",
    "",
    "Speed",
    f"- NeoProphet was faster in total time on {faster} of {len(datasets)} datasets.",
    f"- Aggregate speed-up (upstream / NeoProphet): {geo_mean:.2f}x geometric mean of per-dataset ratios, "
    f"{sum_ratio:.2f}x on summed time.",
    f"- Fit time totals: NeoProphet {fork_t['fit_s'].sum():.2f}s, upstream {up_t['fit_s'].sum():.2f}s.",
    f"- Predict time totals: NeoProphet {fork_t['predict_s'].sum():.2f}s, upstream {up_t['predict_s'].sum():.2f}s.",
]
if runs[FORK]["cross_validation"] and runs[UPSTREAM]["cross_validation"]:
    lines.append(f"- cross_validation on {runs[FORK]['cross_validation']['dataset']}: speed-up {cv_speedup:.2f}x.")
lines += [
    "",
    "Timing differences within roughly 10% are within the run-to-run noise expected on a shared machine; "
    "rerun with a higher REPEATS to tighten them.",
]
print("\n".join(lines))

## 8. Accuracy of all four variants

Same metrics and holdouts as section 4. The tuned model is chosen with cross-validation on the **training part only**; the holdout is never seen by `tune`. Lower is better except for coverage, which should sit near 0.80. A dataset the MLX backend cannot fit is shown as missing, with the reason printed below the table.

In [ ]:
datasets = list(fork_m.index)
ALL_METRICS = ["rmse", "mae", "mase", "smape", "coverage", "interval_width"]


def metric_table(metric: str) -> pd.DataFrame:
    """Rows are datasets, columns are variants. Missing where a variant skipped the dataset."""
    columns = {
        label: pd.Series({d["name"]: d["metrics"][metric] for d in runs[label]["datasets"]}, dtype=float)
        for label in ORDER
    }
    return pd.DataFrame(columns).reindex(datasets)[ORDER]


all_acc = pd.concat({m: metric_table(m) for m in ALL_METRICS}, axis=1)
for label in ORDER:
    for item in runs[label]["skipped"]:
        print(f"{label} skipped {item['name']}: {item['reason']}")
all_acc

In [ ]:
rmse = metric_table("rmse")
ratio_to_upstream = rmse.div(rmse[UPSTREAM], axis=0)
change = (100 * (ratio_to_upstream - 1)).drop(columns=UPSTREAM)
change.loc["geometric mean"] = 100 * (np.exp(np.log(ratio_to_upstream.drop(columns=UPSTREAM)).mean()) - 1)
print("Holdout RMSE change versus upstream, in % (negative is better)")
display(change)

fig, ax = plt.subplots(figsize=(12, 4.4))
shown = [FORK, TUNED, MLX]
x = np.arange(len(datasets))
w = 0.26
for i, label in enumerate(shown):
    ax.bar(x + (i - 1) * w, ratio_to_upstream[label], w, label=label, color=COLORS[label])
ax.axhline(1, color=COLORS["actual"], lw=1, ls="--")
ax.set_yscale("log")
ax.set_xticks(x, datasets, rotation=35, ha="right")
ax.set_ylabel("holdout RMSE / upstream RMSE (log scale)")
ax.set_title("Holdout RMSE relative to upstream: below the dashed line is more accurate")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### Where tuning helps and where it hurts

Tuning picks the best grid point by RMSE over rolling-origin folds inside the training part. The "CV" columns compare that pick with Prophet's own defaults on those folds (so a CV gain is guaranteed by construction, the default point is always in the grid). The "holdout" columns compare the tuned model with the hand-set dataset settings on the final held-out window. A tuned model can look better in CV and still be worse on the holdout, which is the case flagged in the last column. The verdict uses a 2% RMSE band. A third reference, plain Prophet with library defaults for the three searched parameters, is shown too, because "tuning helps" is usually quoted against untuned defaults and the hand-set settings may already be better than those.

In [ ]:
TOL_PCT = 2.0
TUNED_KEYS = ["changepoint_prior_scale", "seasonality_prior_scale", "seasonality_mode"]


def fmt_params(params: dict) -> str:
    return " / ".join(str(params.get(k, "default")) for k in TUNED_KEYS)


def by_name(label: str) -> dict:
    return {d["name"]: d for d in runs[label]["datasets"]}


base_by, tuned_by = by_name(FORK), by_name(TUNED)
rows = {}
for name in datasets:
    t = tuned_by[name]["tuning"]
    defaults_rmse = t["holdout_metrics_prophet_defaults"]["rmse"]
    cv_change = 100 * (t["cv_score_best"] / t["cv_score_prophet_defaults"] - 1)
    holdout_change = 100 * (tuned_by[name]["metrics"]["rmse"] / base_by[name]["metrics"]["rmse"] - 1)
    if holdout_change < -TOL_PCT:
        verdict = "helped"
    elif holdout_change > TOL_PCT:
        verdict = "hurt"
    else:
        verdict = "about the same"
    rows[name] = {
        "hand-set (cps / sps / mode)": fmt_params(base_by[name]["prophet_kwargs"]),
        "tuned pick": fmt_params(t["best_params"]),
        "CV RMSE, Prophet defaults": t["cv_score_prophet_defaults"],
        "CV RMSE, tuned": t["cv_score_best"],
        "CV change %": cv_change,
        "holdout RMSE, Prophet defaults": defaults_rmse,
        "holdout RMSE, hand-set": base_by[name]["metrics"]["rmse"],
        "holdout RMSE, tuned": tuned_by[name]["metrics"]["rmse"],
        "tuned vs Prophet defaults %": 100 * (tuned_by[name]["metrics"]["rmse"] / defaults_rmse - 1),
        "hand-set vs Prophet defaults %": 100 * (base_by[name]["metrics"]["rmse"] / defaults_rmse - 1),
        "holdout change % (tuned vs hand-set)": holdout_change,
        "holdout verdict": verdict,
        "CV said better, holdout worse": bool(cv_change < 0 and holdout_change > TOL_PCT),
    }
tuning_effect = pd.DataFrame(rows).T
for col in ["CV RMSE, Prophet defaults", "CV RMSE, tuned", "CV change %", "holdout RMSE, hand-set",
            "holdout RMSE, tuned", "holdout change % (tuned vs hand-set)", "holdout RMSE, Prophet defaults",
            "tuned vs Prophet defaults %", "hand-set vs Prophet defaults %"]:
    tuning_effect[col] = tuning_effect[col].astype(float)
tuning_effect

### MLX fidelity on the holdout

How far the MLX forecast sits from the CmdStan forecast of the same fork, as the largest absolute `yhat` difference over the holdout divided by the range of the held-out `y`. The accuracy table above shows whether any difference matters for the error metrics.

In [ ]:
mlx_by = by_name(MLX)
fidelity = {}
for name in datasets:
    if name not in mlx_by:
        continue
    a = np.asarray(mlx_by[name]["forecast"]["yhat"])
    b = np.asarray(base_by[name]["forecast"]["yhat"])
    y = np.asarray(base_by[name]["forecast"]["y"])
    fidelity[name] = {
        "max |yhat diff|": float(np.max(np.abs(a - b))),
        "relative to holdout y range": float(np.max(np.abs(a - b)) / (y.max() - y.min())),
        "RMSE change % (MLX vs CmdStan)": 100 * (mlx_by[name]["metrics"]["rmse"] / base_by[name]["metrics"]["rmse"] - 1),
    }
mlx_fidelity = pd.DataFrame(fidelity).T
mlx_fidelity

## 9. Speed of all four variants

Fit and predict are medians over the repeats. For the tuned variant these are the times of the final fit and predict with the chosen parameters; **tuning itself is reported separately** in the next subsection. Because the tuned model may choose different parameters (for example multiplicative seasonality), its fit time can differ from the default model even though the machinery is the same.

In [ ]:
def timings_by_variant(col: str) -> pd.DataFrame:
    columns = {
        label: pd.Series({d["name"]: d["timings_median"][col] for d in runs[label]["datasets"]}, dtype=float)
        for label in ORDER
    }
    return pd.DataFrame(columns).reindex(datasets)[ORDER]


total = timings_by_variant("total_s")
speed_all = pd.concat({c: timings_by_variant(c) for c in ["fit_s", "predict_s", "total_s"]}, axis=1)
speedup_all = total[UPSTREAM].values[:, None] / total.drop(columns=UPSTREAM)
speedup_all.loc["geometric mean"] = np.exp(np.log(speedup_all).mean())
display(speed_all)
print("Total (fit + predict) speed-up, upstream time divided by variant time (above 1 is faster than upstream)")
display(speedup_all)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
x = np.arange(len(datasets))
w = 0.2
for ax, col, title in zip(axes, ["fit_s", "predict_s"], ["Fit time", "Predict time (with uncertainty)"]):
    table = timings_by_variant(col)
    for i, label in enumerate(ORDER):
        ax.bar(x + (i - 1.5) * w, table[label], w, label=label, color=COLORS[label])
    ax.set_yscale("log")
    ax.set_xticks(x, datasets, rotation=35, ha="right")
    ax.set_ylabel("seconds (median, log scale)")
    ax.set_title(title)
axes[0].legend(frameon=False)
fig.suptitle("Per-dataset timing, all variants", y=1.02)
fig.tight_layout()
plt.show()

### Tuning cost

Tuning is a one-off search done before the final fit, so its cost is shown next to the cost of a single default fit and predict. "Tuning / upstream fit+predict" says how many untuned upstream runs the same wall time would buy. The search runs in a process pool across the CPU cores, so the wall time depends on core count.

In [ ]:
cost = pd.DataFrame(
    {
        name: {
            "grid points scored": tuned_by[name]["tuning"]["n_combinations"],
            "CV folds": tuned_by[name]["tuning"]["n_folds"],
            "tuning (s)": tuned_by[name]["tuning"]["median_s"],
            "tuned fit + predict (s)": tuned_by[name]["timings_median"]["total_s"],
            "default fit + predict (s)": base_by[name]["timings_median"]["total_s"],
            "upstream fit + predict (s)": by_name(UPSTREAM)[name]["timings_median"]["total_s"],
        }
        for name in datasets
    }
).T
cost["tuning / upstream fit+predict"] = cost["tuning (s)"] / cost["upstream fit + predict (s)"]
cost["tuning share of tuned pipeline"] = cost["tuning (s)"] / (cost["tuning (s)"] + cost["tuned fit + predict (s)"])
display(cost)

fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(datasets))
ax.barh(y, cost["tuning (s)"], color=COLORS[TUNED], label="tuning")
ax.barh(y, cost["tuned fit + predict (s)"], left=cost["tuning (s)"], color=COLORS[FORK], label="final fit + predict")
ax.set_xscale("log")
ax.set_yticks(y, datasets)
ax.invert_yaxis()
ax.set_xlabel("seconds (log scale)")
ax.set_title("Cost of the tuned pipeline per dataset")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 10. Many-series throughput

N copies of Peyton Manning with small seeded Gaussian noise (a fraction of the series' standard deviation set by `THROUGHPUT_NOISE` in the runner, one generator per copy, so each smaller set is a prefix of the larger ones) are fitted with:

- upstream CmdStan, one Prophet per series, sequentially;
- NeoProphet CmdStan, the same loop;
- NeoProphet `fit_many` on MLX, one batched optimisation. This is given one untimed run on the same batch first so one-off graph compilation is not charged to it.

Only fitting is timed. The MLX fit is also compared with the CmdStan fit of the same series: the largest absolute in-sample `yhat` difference divided by the `y` range of that series, reported as the worst and the median over series. The MLX run measured its own CmdStan reference for that comparison, so its CmdStan time is shown as a cross-check on timing noise.

In [ ]:
def throughput_by_n(label: str) -> dict[int, dict]:
    return {r["n_series"]: r for r in runs[label]["throughput"]}


up_tp, fork_tp, mlx_tp = (throughput_by_n(x) for x in (UPSTREAM, FORK, MLX))
sizes = sorted(mlx_tp)
tp = pd.DataFrame(
    {
        n: {
            "upstream CmdStan (s)": up_tp[n]["cmdstan"]["median_s"],
            "NeoProphet CmdStan (s)": fork_tp[n]["cmdstan"]["median_s"],
            "NeoProphet CmdStan, MLX run (s)": mlx_tp[n]["cmdstan"]["median_s"],
            "NeoProphet MLX fit_many (s)": mlx_tp[n]["mlx"]["median_s"],
            "MLX per series (ms)": 1000 * mlx_tp[n]["mlx"]["per_series_s"],
            "CmdStan per series (ms)": 1000 * fork_tp[n]["cmdstan"]["per_series_s"],
            "MLX speed-up vs upstream": up_tp[n]["cmdstan"]["median_s"] / mlx_tp[n]["mlx"]["median_s"],
            "MLX speed-up vs NeoProphet CmdStan": fork_tp[n]["cmdstan"]["median_s"] / mlx_tp[n]["mlx"]["median_s"],
            "NeoProphet CmdStan speed-up vs upstream": up_tp[n]["cmdstan"]["median_s"] / fork_tp[n]["cmdstan"]["median_s"],
            "max |yhat diff| / y range (worst series)": mlx_tp[n]["mlx"]["max_rel_yhat_diff"],
            "max |yhat diff| / y range (median series)": mlx_tp[n]["mlx"]["median_rel_yhat_diff"],
        }
        for n in sizes
    }
).T
tp.index.name = "series"
tp

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [2, 1.4]})
bars = [
    (UPSTREAM, "upstream CmdStan (s)"),
    (FORK, "NeoProphet CmdStan (s)"),
    (MLX, "NeoProphet MLX fit_many (s)"),
]
x = np.arange(len(sizes))
w = 0.26
for i, (label, col) in enumerate(bars):
    values = tp[col].to_numpy()
    ax.bar(x + (i - 1) * w, values, w, label=label if label != FORK else "NeoProphet CmdStan", color=COLORS[label])
    for xi, v in zip(x + (i - 1) * w, values):
        ax.text(xi, v, f"{v:.1f}s", ha="center", va="bottom", fontsize=8)
ax.set_yscale("log")
ax.set_xticks(x, [f"{n} series" for n in sizes])
ax.set_ylabel("wall time to fit all series (s, log scale)")
ax.set_title("Fitting many series")
ax.legend(frameon=False)

speedups = tp[["MLX speed-up vs upstream", "MLX speed-up vs NeoProphet CmdStan"]]
for i, col in enumerate(speedups.columns):
    ax2.bar(x + (i - 0.5) * 0.36, speedups[col], 0.36, color=[COLORS[UPSTREAM], COLORS[FORK]][i], label=col.replace("MLX speed-up ", ""))
ax2.axhline(1, color=COLORS["actual"], lw=1, ls="--")
ax2.set_xticks(x, [f"{n} series" for n in sizes])
ax2.set_ylabel("CmdStan time / MLX time")
ax2.set_title("MLX speed-up (above 1 is faster)")
ax2.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()

## 11. Summary of the fork's additions

Everything below is computed from the results above, so the wording follows the data.

In [ ]:
def geo(series: pd.Series) -> float:
    return float(np.exp(np.log(series.dropna()).mean()))


tuned_rmse_ratio = rmse[TUNED] / rmse[FORK]
verdicts = tuning_effect["holdout verdict"]
names_by = {v: ", ".join(f"{n} ({tuning_effect.loc[n, 'holdout change % (tuned vs hand-set)']:+.0f}%)" for n in verdicts.index[verdicts == v])
            for v in ("helped", "hurt", "about the same")}
cv_mismatch = list(tuning_effect.index[tuning_effect["CV said better, holdout worse"]])

mlx_rmse_ratio = (rmse[MLX] / rmse[FORK]).dropna()
mlx_skipped = [f"{i['name']} ({i['reason']})" for i in runs[MLX]["skipped"]]

lines = [
    "Tuned (tune on the training part only, then forecast the holdout)",
    f"- Holdout RMSE versus the hand-set NeoProphet default: geometric mean ratio {geo(tuned_rmse_ratio):.3f} "
    f"(below 1 means tuning helped on average).",
    f"- Versus plain Prophet defaults instead of the hand-set settings: geometric mean ratio "
    f"{geo(1 + tuning_effect['tuned vs Prophet defaults %'].astype(float) / 100):.3f} for tuned, "
    f"{geo(1 + tuning_effect['hand-set vs Prophet defaults %'].astype(float) / 100):.3f} for hand-set. "
    f"Tuned beats Prophet defaults on {int((tuning_effect['tuned vs Prophet defaults %'].astype(float) < -TOL_PCT).sum())} "
    f"of {len(datasets)} datasets (by more than {TOL_PCT:.0f}%), is worse on "
    f"{int((tuning_effect['tuned vs Prophet defaults %'].astype(float) > TOL_PCT).sum())}.",
    f"- Helped (more than {TOL_PCT:.0f}% lower RMSE): {names_by['helped'] or 'none'}.",
    f"- Hurt (more than {TOL_PCT:.0f}% higher RMSE): {names_by['hurt'] or 'none'}.",
    f"- Within {TOL_PCT:.0f}%: {names_by['about the same'] or 'none'}.",
    f"- Cross-validation preferred the tuned pick over Prophet's defaults on every dataset by construction, "
    f"yet the holdout got worse on: {', '.join(cv_mismatch) or 'none'}.",
    f"- Tuning wall time {cost['tuning (s)'].min():.1f}s to {cost['tuning (s)'].max():.1f}s per dataset "
    f"({cost['tuning (s)'].sum():.0f}s in total), "
    f"{cost['tuning / upstream fit+predict'].median():.0f}x the median upstream fit+predict.",
    "",
    "MLX single fits",
    f"- Skipped: {'; '.join(mlx_skipped) if mlx_skipped else 'nothing, all datasets are linear growth'}.",
    f"- Holdout RMSE versus NeoProphet CmdStan: geometric mean ratio {geo(mlx_rmse_ratio):.3f}, "
    f"range {mlx_rmse_ratio.min():.3f} to {mlx_rmse_ratio.max():.3f}.",
    f"- Largest yhat difference to CmdStan on a holdout: {mlx_fidelity['relative to holdout y range'].max():.3g} of the y range "
    f"({mlx_fidelity['relative to holdout y range'].idxmax()}).",
    f"- Total fit+predict speed-up versus upstream: geometric mean {speedup_all.loc['geometric mean', MLX]:.2f}x "
    f"(per dataset {speedup_all[MLX].drop('geometric mean').min():.2f}x to {speedup_all[MLX].drop('geometric mean').max():.2f}x).",
    "",
    "Many-series throughput",
]
for n in sizes:
    r = tp.loc[n]
    lines.append(
        f"- {n} series: upstream CmdStan {r['upstream CmdStan (s)']:.1f}s, NeoProphet CmdStan "
        f"{r['NeoProphet CmdStan (s)']:.1f}s, MLX fit_many {r['NeoProphet MLX fit_many (s)']:.2f}s "
        f"({r['MLX speed-up vs upstream']:.1f}x vs upstream, {r['MLX speed-up vs NeoProphet CmdStan']:.1f}x vs NeoProphet CmdStan). "
        f"MLX vs CmdStan yhat: worst series {r['max |yhat diff| / y range (worst series)']:.2%} of the y range, "
        f"median series {r['max |yhat diff| / y range (median series)']:.2%}."
    )
lines += [
    "",
    "Reading these numbers",
    "- One holdout per dataset: a tuned model that wins or loses by a few percent says little. Only large gaps are worth acting on.",
    "- Throughput compares fitting only, on near-identical series; real portfolios of different lengths and regressors batch less evenly.",
    "- MLX results are MAP fits from a different optimiser, so small yhat differences from CmdStan are expected.",
    "- Timing differences within roughly 10% are within the run-to-run noise expected on a shared machine.",
]
print("\n".join(lines))

That is the end of the series. Back to [00. Quick start](00_quick_start.ipynb).